In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns.fpgrowth import fpgrowth
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
df=pd.read_csv('datasets//mercc_cleaned.csv')

In [3]:
# Set hyper-paramater for machine learning
MIN_SUPPORT = 0.001
MIN_THRESH = 0.01 # For report generation

In [4]:
# Feature selection 

df = df[['session_id', 'event_id', 'c0_name', 'c0_id', 'c1_name', 'c1_id']]

# Bucketing based on user action

# Completed purchase - level 0
bask_comp_purchase_zero = df.loc[df['event_id'] == 'buy_comp', ['session_id','c0_name']]

# Completed purchase - level 1
bask_comp_purchase_one = df.loc[df['event_id'] == 'buy_comp', ['session_id','c1_name']]
bask_comp_purchase_one = bask_comp_purchase_one.loc[bask_comp_purchase_one['c1_name'] != 'unknown',:]

# Added to cart - level 0
bask_add_cart_zero = df.loc[df['event_id'] == 'item_add_to_cart_tap', ['session_id','c0_name']]

# Added to cart - level 1
bask_add_cart_one = df.loc[df['event_id'] == 'item_add_to_cart_tap', ['session_id','c1_name']]
bask_add_cart_one = bask_add_cart_one.loc[bask_add_cart_one['c1_name'] != 'unknown',:]

# Item view and Item like is also consideration, but it's low priority

In [5]:
def melt_dataset(basket_df, categ_name, groupby='session_id'):
    '''
        basket_df: Target dataframe
        categ_name: column name of the target category, must be in string
        group_by(Optional): column name of the grouping variable

        Take a dataset (must be in wide format) and column names as an input 
        then turn into long format ready for machine learning
    '''
    basket_df = basket_df.groupby(groupby)[categ_name].apply(list).to_list()

    trans_encoder = TransactionEncoder()

    trans_encoder.fit(basket_df)

    enctrans_basket_df = trans_encoder.transform(basket_df)

    return pd.DataFrame(enctrans_basket_df, columns=trans_encoder.columns_)


bask_comp_purchase_zero_proc = melt_dataset(bask_comp_purchase_zero, 'c0_name')
bask_comp_purchase_one_proc = melt_dataset(bask_comp_purchase_one, 'c1_name')

bask_add_cart_zero_proc = melt_dataset(bask_add_cart_zero, 'c0_name')
bask_add_cart_one_proc = melt_dataset(bask_add_cart_one, 'c1_name')

In [20]:
# This is only for explanation purposes
'''
enc = pd.get_dummies(bask_comp_purchase_zero['c0_name'], prefix='c0_name')
new = pd.concat([bask_comp_purchase_zero, enc], axis=1).drop(columns=['c0_name'])

new.groupby('session_id').max()
'''

,c0_name_Arts & Crafts,c0_name_Beauty,c0_name_Books,c0_name_Electronics,c0_name_Garden & Outdoor,c0_name_Handmade,c0_name_Home,c0_name_Kids,c0_name_Men,c0_name_Office,c0_name_Other,c0_name_Pet Supplies,c0_name_Sports & outdoors,c0_name_Tools,c0_name_Toys & Collectibles,c0_name_Vintage & collectibles,c0_name_Women
session_id,,,,,,,,,,,,,,,,,
003face7ac56f5e3b33a00de303dc22a8315a2418195d7c835830374d073cb8e,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
006c0f36fb966f6f474fb73f4505f387d3a7c4911925aeb0ed2a1b0602394739,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False
00981bd0e3f6049a4d73534e58efa3e91b48b24f1f00becf893770464a8070f2,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
00de2d4f2045b5ccb888851f716923323ca3724ad6d9b1e27cec1c6348f76f0a,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
00f303fcff39927d08d744fc4e21da1a54ecd568af120054f0ab57548299aaf2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ff51ca2ff091b8e7fbdea889fee871f2ba3d2bc9e55955dbf9b2f813939aa676,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
ff7d4841435d173cbf1d5d539eeacb6555baa1a8ebdf166ff32eedcc468d83f1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
ff825c41fefea65f404364c9a7df4aaf5e3a5f14ce6b09460d3408b494d6b02e,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False


In [6]:
# Model Training
freq_bask_comp_purchase_zero = fpgrowth(bask_comp_purchase_zero_proc, min_support=MIN_SUPPORT, use_colnames=True)
freq_bask_comp_purchase_one = fpgrowth(bask_comp_purchase_one_proc, min_support=MIN_SUPPORT, use_colnames=True)

freq_bask_add_cart_zero = fpgrowth(bask_add_cart_zero_proc, min_support=MIN_SUPPORT, use_colnames=True)
freq_bask_add_cart_one = fpgrowth(bask_add_cart_one_proc, min_support=MIN_SUPPORT, use_colnames=True)

In [7]:
# Creating report

machine_dict = {
    "comp_purchase_zero": freq_bask_comp_purchase_zero,
    "comp_purchase_one": freq_bask_comp_purchase_one,
    "add_cart_zero": freq_bask_add_cart_zero,
    "add_cart_one": freq_bask_add_cart_one
}

for name, data in machine_dict.items():
    # Generate rules
    rules = association_rules(data, metric='lift', min_threshold=MIN_THRESH)
    
    # Sort and export
    rules.sort_values(by='lift', ascending=False)\
         .to_csv(f'results\\{name}_association_rule_report.csv', index=False)
    
    print(f"Exported: {name}_association_rule_report.csv")

Exported: comp_purchase_zero_association_rule_report.csv
Exported: comp_purchase_one_association_rule_report.csv
Exported: add_cart_zero_association_rule_report.csv
Exported: add_cart_one_association_rule_report.csv


In [8]:
# interpretation of the result
'''
There's a strong disassociation between the products
meaning placing or selling products in bundle at C2C marketplace, will unlikely to be bought together
Some product are unlikely to be bought together
'''

"\nThere's a strong disassociation between the products\nmeaning placing or selling products in bundle at C2C marketplace, will unlikely to be bought together\nSome product are unlikely to be bought together\n"